# 03 · 사용자 집계와 SRM

**핵심 실행 9분** · 좋아 보이는 지표를 해석하기 전에 데이터의 신뢰성을 확인합니다.

준비: 저장소 터미널에서 `python -m pip install -r requirements/week12-online.txt` 실행.
VS Code 오른쪽 위 **Select Kernel → 저장소 .venv의 Python 3.11**을 선택합니다.
다른 Notebook의 변수나 실행 순서를 요구하지 않습니다. 이 파일의 셀은 위에서 아래로 실행하세요.

**출처 구분:** CSV는 교수 작성 합성 사용자 데이터입니다. FastAPI 요청은 실제 실습 서비스 실행이며 실제 고객 실험이 아닙니다.
05번의 Ollama만 실제 로컬 모델을 호출합니다. Docker·클라우드 계정·유료 API는 필수가 아닙니다.

## 코드 셀 실행 지도

| 코드 셀 | 실행 | 확인할 결과 |
|---:|---|---|
| 1 | 환경 준비 | Python 3.11과 설정 버전 |
| 2 | 종료된 합성 스냅샷 | 2400명 중 A 1196명·B 1204명입니다 |
| 3 | 집계 계약 검증 | 중복 사용자를 추가하면 INVALID입니다 |
| 4 | 정상 배정과 비율 불일치 | 정상 p≈0 |
| 5 | 원래 배정과 수집 결과 대조 | B 602명 누락, 효과 구간을 출력하지 않고 해석을 중단합니다 |
| 6 | 성공 응답만 남기는 함정 | 실패를 제거한 비율은 정의가 바뀌었습니다 |
| 7 | 관찰 지표와 사람의 행동 | 서로 다른 두 비율이 나옵니다 |

In [ ]:
from pathlib import Path
import sys, json, os
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "week12/lab/analysis.py").exists()), None)
if ROOT is None:
    raise RuntimeError("저장소 폴더를 VS Code로 열고 실행하세요.")
if sys.version_info[:2] != (3, 11):
    raise RuntimeError("저장소 .venv의 Python 3.11 커널을 선택하세요.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import pandas as pd
from IPython.display import display, HTML
from week12.lab.experiment import DEFAULT, assign
from week12.lab.analysis import analyze
from week12.lab.simulate import load_units
print("준비 완료:", sys.version.split()[0], "실험 설정", DEFAULT.version)


## 코드 셀 2 · 종료된 합성 스냅샷

가상 14일 실험의 사용자별 첫 요청 결과입니다. 합성 데이터 생성 규칙은 공개 코드에 있습니다.

In [ ]:
data = load_units("healthy")
display(data.head())
metadata = json.loads((ROOT/"week12/lab/data/metadata.json").read_text())
print(metadata["notes"])

**확인과 해석:** 2400명 중 A 1196명·B 1204명입니다.

**직접 설명:** 이 데이터로 실제 제품 효과를 주장하면 왜 잘못인가요?

## 코드 셀 3 · 집계 계약 검증

결측·중복·잘못된 버전은 계산 전에 막습니다. 무조건 dropna로 지우지 않습니다.

In [ ]:
from week12.lab.analysis import validate_units
assert validate_units(data) == []
duplicate = pd.concat([data,data.iloc[[0]]],ignore_index=True)
print(validate_units(duplicate))
assert analyze(duplicate)["decision"] == "INVALID"

**확인과 해석:** 중복 사용자를 추가하면 INVALID입니다.

**직접 설명:** 반복 요청을 사용자가 늘어난 것으로 세면 신뢰구간에 어떤 문제가 생기나요?

## 코드 셀 4 · 정상 배정과 비율 불일치

SRM은 Sample Ratio Mismatch, 설정한 배정 비율과 관찰한 사용자 비율의 불일치입니다. 원인을 알려주는 판결이 아니라 진단 신호입니다.

In [ ]:
from week12.lab.analysis import srm_test
for name in ["healthy","srm"]:
    frame = load_units(name)
    n = frame.variant.value_counts()
    result = srm_test(int(n["A"]),int(n["B"]))
    print(name,n.to_dict(),result)

**확인과 해석:** 정상 p≈0.870, 누락 사례 p는 매우 작습니다.

**직접 설명:** B의 효과가 좋아 보이더라도 먼저 무엇을 멈춰야 하나요?

## 코드 셀 5 · 원래 배정과 수집 결과 대조

이 사례는 B 사용자 일부를 의도적으로 누락한 예시입니다. 배정 로직이 정확해도 수집·조인·필터에서 편향이 생깁니다.

In [ ]:
broken = load_units("srm")
missing = data[~data.visitor_id.isin(broken.visitor_id)]
display(missing.groupby("variant").size().rename("누락 사용자"))
report = analyze(broken)
print(report["decision"], report["reasons"])
assert report["effect"] is None

**확인과 해석:** B 602명 누락, 효과 구간을 출력하지 않고 해석을 중단합니다.

**직접 설명:** 누락된 사람은 무작위 일부일까요, 특정 결과를 경험한 사람일까요?

## 코드 셀 6 · 성공 응답만 남기는 함정

실패 사용자를 지우면 관찰 대상이 결과에 의해 바뀝니다. 오류는 분모에 남기고 운영 기준으로도 봅니다.

In [ ]:
all_users = data.groupby("variant").resolved.mean()
only_ok = data[data.error==0].groupby("variant").resolved.mean()
display(pd.DataFrame({"진입 사용자 전체":all_users,"성공 응답만":only_ok}))
display(data.groupby("variant")[["error","feedback_received"]].mean())

**확인과 해석:** 실패를 제거한 비율은 정의가 바뀌었습니다. 비슷한 숫자여도 같은 지표가 아닙니다.

**직접 설명:** ‘답변한 사람 중 만족 비율’과 ‘전체 진입자 중 해결 확인율’의 분모는 어떻게 다른가요?

## 코드 셀 7 · 관찰 지표와 사람의 행동

피드백 미응답은 실제 해결 실패를 뜻하지 않습니다. 수집률과 별도 사용성 관찰을 함께 봅니다.

In [ ]:
observed = data[data.feedback_received==1].groupby("variant").resolved.mean()
display(pd.DataFrame({"전체 사용자 해결 확인율":all_users,"피드백 응답자 내 비율":observed}))
print("응답자만의 만족도와 전체 사용자 지표를 혼동하지 않습니다.")

**확인과 해석:** 서로 다른 두 비율이 나옵니다.

**직접 설명:** B에서 피드백 버튼이 더 잘 보이도록 바꾸면 어떤 측정 변화가 생길까요?

## 복습과 다음 단계

MSN의 이미지 카드 실험 사례와 비교해 보세요. 결과에 영향받은 사용자만 필터링되면 판단 방향까지 바뀔 수 있습니다. 다음 Notebook은 유효한 데이터의 불확실성과 운영 기준을 함께 봅니다.

실행 오류는 Kernel을 다시 시작한 뒤 위에서부터 실행해 확인합니다. `ModuleNotFoundError`는 설치 터미널과 Notebook 커널이 같은 `.venv`인지 확인하세요.
실험 결과는 `week12/lab/reports/`에 저장하고 공개 GitHub에 학생 답안·개인정보를 올리지 않습니다.
[실행 안내](../WALKTHROUGH.md) · [100분 강의안](https://smilesjcha.github.io/llmops-16week/week12/lecture/) · [9–12주차 과제](https://smilesjcha.github.io/llmops-16week/week12/assignment/)
